# NixLMs 0.7 — Google Colab
Notebook oficial sincronizado com `main`. Inclui memória episódica, World Model, objetivos, atenção, recuperação semântica e integração opcional com Ollama. O núcleo funciona em CPU; Ollama e download de modelo são opcionais.

In [ ]:
from pathlib import Pathimport subprocess, sysREPO_DIR = Path('/content/NixLMs')if not REPO_DIR.exists():    subprocess.run(['git', 'clone', 'https://github.com/SANTZSPAXX/NixLMs.git', str(REPO_DIR)], check=True)else:    subprocess.run(['git', '-C', str(REPO_DIR), 'fetch', 'origin', 'main'], check=True)    subprocess.run(['git', '-C', str(REPO_DIR), 'checkout', 'main'], check=True)    subprocess.run(['git', '-C', str(REPO_DIR), 'pull', '--ff-only', 'origin', 'main'], check=True)%cd /content/NixLMs# Descarta módulos importados anteriormente para evitar código Python em cache.for module_name in list(sys.modules):    if module_name == 'nixlms_core' or module_name.startswith('nixlms_core.'):        del sys.modules[module_name]previous_ia = globals().get('ia')if previous_ia is not None:    previous_ia.scheduler.stop()globals().pop('ia', None)from nixlms_core import NixLMsNeuralCoreimport json, osprint('Python:', sys.version)print('Checkout:', subprocess.check_output(['git', 'rev-parse', '--short', 'HEAD'], text=True).strip())print('Código importado de:', __import__('nixlms_core').__file__)

## Persistência entre sessões (Google Drive)Execute esta célula para guardar memória, pesos, objetivos e dados no Drive e iniciar o runtime cognitivo com diagnóstico e aprendizagem ociosa. Se preferir armazenamento temporário da sessão, defina `USE_GOOGLE_DRIVE = False`.

In [ ]:
from pathlib import PathUSE_GOOGLE_DRIVE = Trueif USE_GOOGLE_DRIVE:    from google.colab import drive    drive.mount('/content/drive')    DATA_DIR = Path('/content/drive/MyDrive/NixLMs/data')else:    DATA_DIR = Path('/content/NixLMs/data')DATA_DIR.mkdir(parents=True, exist_ok=True)previous_ia = globals().get('ia')if previous_ia is not None:    previous_ia.scheduler.stop()ia = NixLMsNeuralCore(data_dir=DATA_DIR)RUNTIME = ia.start_runtime()print('Autodiagnóstico:', RUNTIME['diagnostics']['status'])print('Scheduler iniciado:', RUNTIME['scheduler_started'])print('Aprendizagem ociosa:', json.dumps(RUNTIME['idle_learning'], ensure_ascii=False))if not ia.learning.dataset.load():    print('Dataset inicial:', ia.learning.dataset.import_jsonl('datasets/starter_ptbr.jsonl'))print(json.dumps(ia.status(), ensure_ascii=False, indent=2, default=str))

## Ollama local (opcional)
Execute esta célula para configurar geração local. Ela pede confirmações separadas antes de instalar Ollama, iniciar o serviço e baixar `qwen2.5:0.5b`. Nenhuma dessas operações ocorre sem consentimento explícito.


In [ ]:
# O gerenciador compartilhado com a CLI não executa ações sem autorização.
import json

if ia.ollama.executable() is None:
    if input("Ollama não está instalado. Autoriza instalar? No Linux isso pode atualizar índices apt, instalar zstd e executar o instalador oficial remoto de ollama.com. [s/N] ").strip().lower() in {"s", "sim", "y", "yes"}:
        result = ia.ollama.install()
        print("Instalação:", result)
        if not result.get("ok"):
            raise RuntimeError(result.get("error", "Falha desconhecida na instalação."))
    else:
        print("Instalação não autorizada; nenhuma alteração foi realizada.")

ia.language.refresh_models()
if ia.ollama.executable() and not ia.language.reachable:
    if input("Autoriza iniciar o serviço Ollama local? [s/N] ").strip().lower() in {"s", "sim", "y", "yes"}:
        result = ia.ollama.start_service()
        print("Inicialização:", result)
        if not result.get("ok"):
            raise RuntimeError(result.get("error", "Falha desconhecida ao iniciar o serviço."))
    else:
        print("Inicialização não autorizada.")

ia.language.refresh_models()
TARGET_MODEL = "qwen2.5:0.5b"
if ia.language.reachable and TARGET_MODEL not in ia.language.installed_models:
    if input("Autoriza baixar qwen2.5:0.5b? [s/N] ").strip().lower() in {"s", "sim", "y", "yes"}:
        result = ia.ollama.pull_model(TARGET_MODEL)
        print("Download/verificação:", result)
        if not result.get("ok"):
            raise RuntimeError(result.get("error", "Falha desconhecida no download."))
    else:
        print("Download não autorizado; nenhum modelo foi baixado.")

ia.language.refresh_models()
if TARGET_MODEL in ia.language.installed_models:
    ia.language.select_model(TARGET_MODEL)
    test_answer = ia.language.generate("Responda somente com a palavra OK.")
    if test_answer and ia.language.last_used_model == TARGET_MODEL:
        print("SUCESSO: geração confirmada pelo modelo", ia.language.last_used_model)
        print("Resposta de teste:", test_answer)
        print("Geração verificada em:", ia.language.last_generation)
    else:
        raise RuntimeError("Não foi possível confirmar o uso do modelo: " + str(ia.language.last_error))
else:
    print("Qwen não está disponível. Status:", json.dumps(ia.ollama.status(), ensure_ascii=False, indent=2))


## Conversa e comandos
Use `/llm` para verificar o serviço, `/llm usar NOME` para selecionar um modelo instalado, `/rollback ID` para restaurar um checkpoint aceito, `/positivo` ou `/negativo` para avaliar a última resposta, `/ensinar pergunta => resposta` para registrar conhecimento e `/status` para inspecionar os módulos. O feedback afeta a recuperação episódica; não altera pesos neurais automaticamente. O split `validation` calibra o limiar LIF sem participar do treino; use `/calibrar-lif` depois de `/treinar`. `/avaliar` mede o fluxo integrado offline sem registrar episódios de teste; `/avaliar-llm` inclui o modelo Ollama opcional.

In [ ]:
def nix_chat():    print('NixLMs 0.7 — digite /ajuda ou /sair')    while True:        message = input('Você: ').strip()        low = message.lower()        if low in {'/sair', '/exit', '/quit'}:            print('Sessão encerrada.')            break        if not message:            continue        if low == '/ajuda':            print('/status | /diagnostico | /llm | /llm usar NOME | /positivo | /negativo | /mapa | /avaliar | /treinar [epocas] | /ensinar pergunta => resposta | /sair')        elif low == '/status':            print(json.dumps(ia.status(), ensure_ascii=False, indent=2, default=str))        elif low == '/diagnostico':            print(json.dumps({'checkout': os.popen('git rev-parse --short HEAD').read().strip(), 'llm': ia.language.status(), 'last_trace': ia.last_trace}, ensure_ascii=False, indent=2, default=str))        elif low == '/llm':            available = ia.language.available()            print('Ollama disponível:', available)            print(ia.language.status())        elif low.startswith('/llm usar '):            try:                print('Modelo selecionado:', ia.language.select_model(message[len('/llm usar '):].strip()))            except ValueError as exc:                print('Falha:', exc)        elif low in {'/positivo', '/negativo'}:            ok = ia.feedback(ia.last_question, low == '/positivo')            print('Feedback registrado.' if ok else 'Não há episódio anterior correspondente.')        elif low == '/mapa':            print({'nodes': len(ia.mental_map.nodes), 'edges': len(ia.mental_map.edges), 'related': ia.mental_map.associate(ia.last_question, 10)})        elif low.startswith('/importar '):            try:                imported = ia.learning.dataset.import_jsonl(message[len('/importar '):].strip())                if imported:                    ia.invalidate_lif_calibration()                print('Registros importados:', imported)            except (OSError, ValueError) as exc:                print('Falha na importação:', exc)        elif low == '/calibrar-decoder':            try:                print(ia.learning.calibrate_decoder_confidence())            except ValueError as exc:                print('Falha na calibração do decoder:', exc)        elif low == '/calibrar-lif':            print(ia.learning.calibrate_lif_threshold())        elif low == '/avaliar-llm':            print(ia.learning.evaluate(include_llm=True))        elif low == '/avaliar':            print(ia.learning.evaluate())        elif low.startswith('/rollback '):            checkpoint_id = message.split(None, 1)[1].strip()            try:                print(ia.learning.restore_incremental_checkpoint(checkpoint_id))            except (OSError, ValueError) as exc:                print('Falha no rollback:', exc)        elif low.startswith('/treinar'):            parts = low.split()            print(ia.learning.train_incremental(int(parts[1]) if len(parts) > 1 else 1))        elif low.startswith('/ensinar '):            payload = message[len('/ensinar '):]            if '=>' not in payload:                print('Use /ensinar pergunta => resposta')            else:                question, answer = payload.split('=>', 1)                ia.learn(question, answer, add_to_dataset=True)                print('Conhecimento salvo.')        else:            print('NixLMs:', ia.respond(message, use_language=True))nix_chat()ia.scheduler.stop()print('Agendador da NixLMs encerrado.')

## Testes automatizados
A suíte usa mocks para serviços externos e não exige Ollama. A execução confirma o comportamento do código no runtime atual do Colab.

In [ ]:
!python -m unittest discover -s tests -v